In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder.appName("Spark Functions").enableHiveSupport().getOrCreate()
print("Driver Python:", sys.executable)
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 09:12:52 WARN Utils: Your hostname, Darviks-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/06/11 09:12:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 09:12:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Driver Python: /Users/darvikkunalbanda/DataEngineering/.venv/bin/python
Spark: 4.1.2


## String Functions

In [2]:
df = spark.read.parquet("/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/orders.parquet")
df.show(5,False)

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|order_date         |status   |
+--------+------------+--------+----------+-------------------+---------+
|ORD-010 |Doohickey C |3       |29.99     |2025-06-10 12:00:00|delivered|
|ORD-004 |Doohickey C |2       |29.99     |2025-03-12 16:45:00|shipped  |
|ORD-007 |Doohickey C |4       |29.99     |2025-05-02 13:20:00|shipped  |
|ORD-005 |Gadget B    |1       |49.99     |2025-04-01 11:00:00|cancelled|
|ORD-006 |Widget A    |10      |19.99     |2025-04-18 08:30:00|delivered|
+--------+------------+--------+----------+-------------------+---------+
only showing top 5 rows


### Using SQL

In [3]:
#upper case
df.createOrReplaceTempView("order_table")
spark.sql("select product_name, upper(product_name) as prd_nm from order_table").show(10,False)

+------------+-----------+
|product_name|prd_nm     |
+------------+-----------+
|Doohickey C |DOOHICKEY C|
|Doohickey C |DOOHICKEY C|
|Doohickey C |DOOHICKEY C|
|Gadget B    |GADGET B   |
|Widget A    |WIDGET A   |
|Gadget B    |GADGET B   |
|Widget A    |WIDGET A   |
|Widget A    |WIDGET A   |
|Gadget B    |GADGET B   |
|Widget A    |WIDGET A   |
+------------+-----------+



In [4]:
#lower case
spark.sql("select status, lower(status) as status_lower from order_table").show(5,False)

+---------+------------+
|status   |status_lower|
+---------+------------+
|delivered|delivered   |
|shipped  |shipped     |
|shipped  |shipped     |
|cancelled|cancelled   |
|delivered|delivered   |
+---------+------------+
only showing top 5 rows


### Using DataFrame

In [5]:
#upper case 
from pyspark.sql.functions import upper , col
df.withColumn('product_name_upper',upper(col("product_name"))).select("product_name","product_name_upper").show(10,False)

+------------+------------------+
|product_name|product_name_upper|
+------------+------------------+
|Doohickey C |DOOHICKEY C       |
|Doohickey C |DOOHICKEY C       |
|Doohickey C |DOOHICKEY C       |
|Gadget B    |GADGET B          |
|Widget A    |WIDGET A          |
|Gadget B    |GADGET B          |
|Widget A    |WIDGET A          |
|Widget A    |WIDGET A          |
|Gadget B    |GADGET B          |
|Widget A    |WIDGET A          |
+------------+------------------+



In [6]:
#lower case
from pyspark.sql.functions import lower
df.withColumn('product_name_lower',lower(col('product_name'))).select("product_name",'product_name_lower').show(5,False)

+------------+------------------+
|product_name|product_name_lower|
+------------+------------------+
|Doohickey C |doohickey c       |
|Doohickey C |doohickey c       |
|Doohickey C |doohickey c       |
|Gadget B    |gadget b          |
|Widget A    |widget a          |
+------------+------------------+
only showing top 5 rows


In [7]:
#concat
from pyspark.sql.functions import concat_ws

df.withColumn("full_order", concat_ws(' | ', col('order_id'), col('product_name'))).select('order_id','product_name','full_order').show(5,False)

+--------+------------+---------------------+
|order_id|product_name|full_order           |
+--------+------------+---------------------+
|ORD-010 |Doohickey C |ORD-010 | Doohickey C|
|ORD-004 |Doohickey C |ORD-004 | Doohickey C|
|ORD-007 |Doohickey C |ORD-007 | Doohickey C|
|ORD-005 |Gadget B    |ORD-005 | Gadget B   |
|ORD-006 |Widget A    |ORD-006 | Widget A   |
+--------+------------+---------------------+
only showing top 5 rows


In [8]:
#length
from pyspark.sql.functions import length
df.withColumn('product_name_length',length(col('product_name'))).show()

+--------+------------+--------+----------+-------------------+---------+-------------------+
|order_id|product_name|quantity|unit_price|         order_date|   status|product_name_length|
+--------+------------+--------+----------+-------------------+---------+-------------------+
| ORD-010| Doohickey C|       3|     29.99|2025-06-10 12:00:00|delivered|                 11|
| ORD-004| Doohickey C|       2|     29.99|2025-03-12 16:45:00|  shipped|                 11|
| ORD-007| Doohickey C|       4|     29.99|2025-05-02 13:20:00|  shipped|                 11|
| ORD-005|    Gadget B|       1|     49.99|2025-04-01 11:00:00|cancelled|                  8|
| ORD-006|    Widget A|      10|     19.99|2025-04-18 08:30:00|delivered|                  8|
| ORD-002|    Gadget B|       1|     49.99|2025-02-20 14:15:00|delivered|                  8|
| ORD-009|    Widget A|       1|     19.99|2025-06-01 10:00:00|  shipped|                  8|
| ORD-003|    Widget A|       5|     19.99|2025-03-05 09:00:

In [9]:
#substring
from pyspark.sql.functions import substring

df.withColumn('product_name_substring',col('product_name').substr(-1,1)).select('product_name','product_name_substring').show()

+------------+----------------------+
|product_name|product_name_substring|
+------------+----------------------+
| Doohickey C|                     C|
| Doohickey C|                     C|
| Doohickey C|                     C|
|    Gadget B|                     B|
|    Widget A|                     A|
|    Gadget B|                     B|
|    Widget A|                     A|
|    Widget A|                     A|
|    Gadget B|                     B|
|    Widget A|                     A|
+------------+----------------------+



## Arthematic Functions

In [10]:
df.show()
df.printSchema()

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|         order_date|   status|
+--------+------------+--------+----------+-------------------+---------+
| ORD-010| Doohickey C|       3|     29.99|2025-06-10 12:00:00|delivered|
| ORD-004| Doohickey C|       2|     29.99|2025-03-12 16:45:00|  shipped|
| ORD-007| Doohickey C|       4|     29.99|2025-05-02 13:20:00|  shipped|
| ORD-005|    Gadget B|       1|     49.99|2025-04-01 11:00:00|cancelled|
| ORD-006|    Widget A|      10|     19.99|2025-04-18 08:30:00|delivered|
| ORD-002|    Gadget B|       1|     49.99|2025-02-20 14:15:00|delivered|
| ORD-009|    Widget A|       1|     19.99|2025-06-01 10:00:00|  shipped|
| ORD-003|    Widget A|       5|     19.99|2025-03-05 09:00:00|  pending|
| ORD-008|    Gadget B|       2|     49.99|2025-05-25 15:10:00|  pending|
| ORD-001|    Widget A|       3|     19.99|2025-01-15 10:30:00|  shipped|
+--------+------------+--------+------

In [11]:
df = df.withColumn("unit_price",col('unit_price').cast(IntegerType()))
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [12]:
#add
df.withColumn('add_col',col('quantity')+col('unit_price')).select('order_id','quantity','unit_price','add_col').show()

+--------+--------+----------+-------+
|order_id|quantity|unit_price|add_col|
+--------+--------+----------+-------+
| ORD-010|       3|        29|     32|
| ORD-004|       2|        29|     31|
| ORD-007|       4|        29|     33|
| ORD-005|       1|        49|     50|
| ORD-006|      10|        19|     29|
| ORD-002|       1|        49|     50|
| ORD-009|       1|        19|     20|
| ORD-003|       5|        19|     24|
| ORD-008|       2|        49|     51|
| ORD-001|       3|        19|     22|
+--------+--------+----------+-------+



In [13]:
#sub
df.withColumn('sub_col',col('unit_price') - col('quantity')).select('order_id','quantity','unit_price','sub_col').show()

+--------+--------+----------+-------+
|order_id|quantity|unit_price|sub_col|
+--------+--------+----------+-------+
| ORD-010|       3|        29|     26|
| ORD-004|       2|        29|     27|
| ORD-007|       4|        29|     25|
| ORD-005|       1|        49|     48|
| ORD-006|      10|        19|      9|
| ORD-002|       1|        49|     48|
| ORD-009|       1|        19|     18|
| ORD-003|       5|        19|     14|
| ORD-008|       2|        49|     47|
| ORD-001|       3|        19|     16|
+--------+--------+----------+-------+



In [14]:
#multiply
df.withColumn('mul_col',col('unit_price')*col('quantity')).select('order_id','quantity','unit_price','mul_col').show()

+--------+--------+----------+-------+
|order_id|quantity|unit_price|mul_col|
+--------+--------+----------+-------+
| ORD-010|       3|        29|     87|
| ORD-004|       2|        29|     58|
| ORD-007|       4|        29|    116|
| ORD-005|       1|        49|     49|
| ORD-006|      10|        19|    190|
| ORD-002|       1|        49|     49|
| ORD-009|       1|        19|     19|
| ORD-003|       5|        19|     95|
| ORD-008|       2|        49|     98|
| ORD-001|       3|        19|     57|
+--------+--------+----------+-------+



In [15]:
#divide
df.withColumn('div_col',col('unit_price')/col('quantity')).select('order_id','quantity','unit_price','div_col').show()

+--------+--------+----------+-----------------+
|order_id|quantity|unit_price|          div_col|
+--------+--------+----------+-----------------+
| ORD-010|       3|        29|9.666666666666666|
| ORD-004|       2|        29|             14.5|
| ORD-007|       4|        29|             7.25|
| ORD-005|       1|        49|             49.0|
| ORD-006|      10|        19|              1.9|
| ORD-002|       1|        49|             49.0|
| ORD-009|       1|        19|             19.0|
| ORD-003|       5|        19|              3.8|
| ORD-008|       2|        49|             24.5|
| ORD-001|       3|        19|6.333333333333333|
+--------+--------+----------+-----------------+



In [16]:
#modulus
df.withColumn('mod_col',col('unit_price')%col('quantity')).select('order_id','quantity','unit_price','mod_col').show()

+--------+--------+----------+-------+
|order_id|quantity|unit_price|mod_col|
+--------+--------+----------+-------+
| ORD-010|       3|        29|      2|
| ORD-004|       2|        29|      1|
| ORD-007|       4|        29|      1|
| ORD-005|       1|        49|      0|
| ORD-006|      10|        19|      9|
| ORD-002|       1|        49|      0|
| ORD-009|       1|        19|      0|
| ORD-003|       5|        19|      4|
| ORD-008|       2|        49|      1|
| ORD-001|       3|        19|      1|
+--------+--------+----------+-------+



In [17]:
from pyspark.sql.types import DoubleType
df = df.withColumn("unit_price",col('unit_price').cast(DoubleType()))
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [18]:
#Round
from pyspark.sql.functions import round
df.withColumn("round_val",round(col('unit_price'))).select('order_id','unit_price','round_val').show()

+--------+----------+---------+
|order_id|unit_price|round_val|
+--------+----------+---------+
| ORD-010|      29.0|     29.0|
| ORD-004|      29.0|     29.0|
| ORD-007|      29.0|     29.0|
| ORD-005|      49.0|     49.0|
| ORD-006|      19.0|     19.0|
| ORD-002|      49.0|     49.0|
| ORD-009|      19.0|     19.0|
| ORD-003|      19.0|     19.0|
| ORD-008|      49.0|     49.0|
| ORD-001|      19.0|     19.0|
+--------+----------+---------+



### date functions

In [19]:
df.show()
df.printSchema()

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|         order_date|   status|
+--------+------------+--------+----------+-------------------+---------+
| ORD-010| Doohickey C|       3|      29.0|2025-06-10 12:00:00|delivered|
| ORD-004| Doohickey C|       2|      29.0|2025-03-12 16:45:00|  shipped|
| ORD-007| Doohickey C|       4|      29.0|2025-05-02 13:20:00|  shipped|
| ORD-005|    Gadget B|       1|      49.0|2025-04-01 11:00:00|cancelled|
| ORD-006|    Widget A|      10|      19.0|2025-04-18 08:30:00|delivered|
| ORD-002|    Gadget B|       1|      49.0|2025-02-20 14:15:00|delivered|
| ORD-009|    Widget A|       1|      19.0|2025-06-01 10:00:00|  shipped|
| ORD-003|    Widget A|       5|      19.0|2025-03-05 09:00:00|  pending|
| ORD-008|    Gadget B|       2|      49.0|2025-05-25 15:10:00|  pending|
| ORD-001|    Widget A|       3|      19.0|2025-01-15 10:30:00|  shipped|
+--------+------------+--------+------

In [20]:
# yyyy/mm/dd --> mm/dd/yyyy
from pyspark.sql.functions import to_date , date_format
df.withColumn("standard_format", to_date(col("order_date"),"mm/dd/yyyy")).show()

+--------+------------+--------+----------+-------------------+---------+---------------+
|order_id|product_name|quantity|unit_price|         order_date|   status|standard_format|
+--------+------------+--------+----------+-------------------+---------+---------------+
| ORD-010| Doohickey C|       3|      29.0|2025-06-10 12:00:00|delivered|     2025-06-10|
| ORD-004| Doohickey C|       2|      29.0|2025-03-12 16:45:00|  shipped|     2025-03-12|
| ORD-007| Doohickey C|       4|      29.0|2025-05-02 13:20:00|  shipped|     2025-05-02|
| ORD-005|    Gadget B|       1|      49.0|2025-04-01 11:00:00|cancelled|     2025-04-01|
| ORD-006|    Widget A|      10|      19.0|2025-04-18 08:30:00|delivered|     2025-04-18|
| ORD-002|    Gadget B|       1|      49.0|2025-02-20 14:15:00|delivered|     2025-02-20|
| ORD-009|    Widget A|       1|      19.0|2025-06-01 10:00:00|  shipped|     2025-06-01|
| ORD-003|    Widget A|       5|      19.0|2025-03-05 09:00:00|  pending|     2025-03-05|
| ORD-008|

In [21]:
df.withColumn("std_format",date_format(to_date(col('order_date'),'yyyy-mm-dd'),'dd/MM/yyyy')).select('order_id','status','order_date','std_format').show()

+--------+---------+-------------------+----------+
|order_id|   status|         order_date|std_format|
+--------+---------+-------------------+----------+
| ORD-010|delivered|2025-06-10 12:00:00|10/06/2025|
| ORD-004|  shipped|2025-03-12 16:45:00|12/03/2025|
| ORD-007|  shipped|2025-05-02 13:20:00|02/05/2025|
| ORD-005|cancelled|2025-04-01 11:00:00|01/04/2025|
| ORD-006|delivered|2025-04-18 08:30:00|18/04/2025|
| ORD-002|delivered|2025-02-20 14:15:00|20/02/2025|
| ORD-009|  shipped|2025-06-01 10:00:00|01/06/2025|
| ORD-003|  pending|2025-03-05 09:00:00|05/03/2025|
| ORD-008|  pending|2025-05-25 15:10:00|25/05/2025|
| ORD-001|  shipped|2025-01-15 10:30:00|15/01/2025|
+--------+---------+-------------------+----------+



In [22]:
df.withColumn("standard_format",date_format(col('order_date'),"dd*MM*yyyy")).show()

+--------+------------+--------+----------+-------------------+---------+---------------+
|order_id|product_name|quantity|unit_price|         order_date|   status|standard_format|
+--------+------------+--------+----------+-------------------+---------+---------------+
| ORD-010| Doohickey C|       3|      29.0|2025-06-10 12:00:00|delivered|     10*06*2025|
| ORD-004| Doohickey C|       2|      29.0|2025-03-12 16:45:00|  shipped|     12*03*2025|
| ORD-007| Doohickey C|       4|      29.0|2025-05-02 13:20:00|  shipped|     02*05*2025|
| ORD-005|    Gadget B|       1|      49.0|2025-04-01 11:00:00|cancelled|     01*04*2025|
| ORD-006|    Widget A|      10|      19.0|2025-04-18 08:30:00|delivered|     18*04*2025|
| ORD-002|    Gadget B|       1|      49.0|2025-02-20 14:15:00|delivered|     20*02*2025|
| ORD-009|    Widget A|       1|      19.0|2025-06-01 10:00:00|  shipped|     01*06*2025|
| ORD-003|    Widget A|       5|      19.0|2025-03-05 09:00:00|  pending|     05*03*2025|
| ORD-008|

### aggregate functions
min , max , avg , sum , count

In [23]:
df.show(3,False)
df.printSchema()

from pyspark.sql.functions import min, max , avg , sum , count

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|order_date         |status   |
+--------+------------+--------+----------+-------------------+---------+
|ORD-010 |Doohickey C |3       |29.0      |2025-06-10 12:00:00|delivered|
|ORD-004 |Doohickey C |2       |29.0      |2025-03-12 16:45:00|shipped  |
|ORD-007 |Doohickey C |4       |29.0      |2025-05-02 13:20:00|shipped  |
+--------+------------+--------+----------+-------------------+---------+
only showing top 3 rows
root
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [24]:
df.agg(min('unit_price'),avg('unit_price'),sum('unit_price'),count('unit_price')).show()

+---------------+---------------+---------------+-----------------+
|min(unit_price)|avg(unit_price)|sum(unit_price)|count(unit_price)|
+---------------+---------------+---------------+-----------------+
|           19.0|           31.0|          310.0|               10|
+---------------+---------------+---------------+-----------------+



## joins

In [25]:
customer_df = spark.read.parquet('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/customers_parquet')
orders_df = spark.read.csv('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/orders_csv',header=True)

customer_df.show(3,False)

customer_df.printSchema()

orders_df.show(3,False)

orders_df.printSchema()

+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|name|city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate|Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack|Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |
|C003       |Leo |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 3 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- tier: string (nullable = true)
 |-- score: integer (nullable = true)
 |-- last_order_date: date (nullable = true)
 |-- active: string (nullable = true

In [30]:
#Inner Join

inner_df = customer_df.join(orders_df, customer_df.customer_id==orders_df.customer_id, 'inner').select('name','status')
inner_df.show(5,False)

+----+---------+
|name|status   |
+----+---------+
|Jack|cancelled|
|Jack|pending  |
|Bob |delivered|
|Bob |cancelled|
|Bob |shipped  |
+----+---------+
only showing top 5 rows


In [32]:
# left outter join
left_df = customer_df.join(orders_df, customer_df.customer_id==orders_df.customer_id, 'left')
left_df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+------+-----------------------------+---------+-----------+-------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|order_id|customer_id|product          |quantity|unit_price|total |order_date                   |status   |payment    |channel|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+------+-----------------------------+---------+-----------+-------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |NULL    |NULL       |NULL             |NULL    |NULL      |NULL  |NULL                         |NULL     |NULL       |NULL   |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |ORD-012 |C002       |Whatchamacallit E|5       

In [34]:
#right outter join
r_df = customer_df.join(orders_df, customer_df.customer_id==orders_df.customer_id, 'right')
r_df.show(5,False)

+-----------+------+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+-------+-----------------------------+---------+-----------+--------+
|customer_id|name  |city    |state|signup_date|tier  |score|last_order_date|active|region|order_id|customer_id|product          |quantity|unit_price|total  |order_date                   |status   |payment    |channel |
+-----------+------+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+-------+-----------------------------+---------+-----------+--------+
|C040       |Olivia|Denver  |IL   |2024-03-14 |Bronze|283  |2025-06-11     |false |South |ORD-001 |C040       |Widget A         |15      |65.09     |976.35 |2025-06-19T11:45:44.000+05:30|cancelled|Cash       |Mobile  |
|C010       |Noah  |New York|CO   |2024-01-26 |Gold  |510  |2025-03-03     |true  |East  |ORD-002 |C010       |Widget A     

In [ ]:
#full outter join

f_df = customer_df.join(orders_df, customer_df.customer_id==orders_df.customer_id, 'full')
f_df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+------+-----------------------------+---------+-----------+-------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|order_id|customer_id|product          |quantity|unit_price|total |order_date                   |status   |payment    |channel|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+-----------------+--------+----------+------+-----------------------------+---------+-----------+-------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |NULL    |NULL       |NULL             |NULL    |NULL      |NULL  |NULL                         |NULL     |NULL       |NULL   |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |ORD-011 |C002       |Whatchamacallit E|2       

In [36]:
#cross join

c_df = customer_df.crossJoin(orders_df)
c_df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+--------+--------+----------+------+-----------------------------+---------+-------+-------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|order_id|customer_id|product |quantity|unit_price|total |order_date                   |status   |payment|channel|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+--------+-----------+--------+--------+----------+------+-----------------------------+---------+-------+-------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |ORD-001 |C040       |Widget A|15      |65.09     |976.35|2025-06-19T11:45:44.000+05:30|cancelled|Cash   |Mobile |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |ORD-001 |C040       |Widget A|15      |65.09     |976.35|2025-06-19T11:45:44.000+05:30|cancelled|Ca

## Union

In [37]:
# Union - it combines two dataframes lnd it will not allow duplicates
customer_df.union(orders_df).show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C001|   Kate|  Boston|   NY| 2024-12-09|  Silver|  328|     2025-02-24|  true| North|
|       C002|   Jack|Portland|   NY| 2024-01-03|  Silver|  338|     2025-05-20|  true| South|
|       C003|    Leo| Atlanta|   OR| 2024-04-15|    Gold|  928|     2025-01-25|  true|  West|
|       C004|  Frank|  Austin|   IL| 2024-04-25|    Gold|  204|     2025-01-13|  true|  East|
|       C005|   Noah| Seattle|   TX| 2024-05-26|  Bronze|  847|     2025-04-18|  true|  West|
|       C006|    Bob| Atlanta|   TX| 2024-11-20|    Gold|  691|     2025-02-23|  true| North|
|       C007|   Kate|  Denver|   TX| 2024-02-28|  Silver|  987|     2025-01-13| false|  West|
|       C008|   Kate| Seattle|   IL| 2024-06-12|  Silver|  7

In [38]:
# Union ALL - It combines two dataframes and it allows duplicates.
customer_df.unionAll(orders_df).show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C001|   Kate|  Boston|   NY| 2024-12-09|  Silver|  328|     2025-02-24|  true| North|
|       C002|   Jack|Portland|   NY| 2024-01-03|  Silver|  338|     2025-05-20|  true| South|
|       C003|    Leo| Atlanta|   OR| 2024-04-15|    Gold|  928|     2025-01-25|  true|  West|
|       C004|  Frank|  Austin|   IL| 2024-04-25|    Gold|  204|     2025-01-13|  true|  East|
|       C005|   Noah| Seattle|   TX| 2024-05-26|  Bronze|  847|     2025-04-18|  true|  West|
|       C006|    Bob| Atlanta|   TX| 2024-11-20|    Gold|  691|     2025-02-23|  true| North|
|       C007|   Kate|  Denver|   TX| 2024-02-28|  Silver|  987|     2025-01-13| false|  West|
|       C008|   Kate| Seattle|   IL| 2024-06-12|  Silver|  7